# Study D, Milestone 3
Corrects the Milestone 2 interpretation, separates clone occupancy from switching rate, and derives the lineage sampling-design rule. See `reports/milestone3_decision.md`.

In [1]:
import json, os, sys, numpy as np, pandas as pd
sys.path.insert(0, '..')
pd.set_option('display.width', 220)
T = '../results/tables/'
have = lambda f: os.path.exists(T + f)

## 1. The audit (U1): the plateau is invariant to the switching rate

In [2]:
expit = lambda x: 1/(1+np.exp(-x)); logit = lambda p: np.log(p/(1-p))
pi = np.array([0.2, 0.8])
pd.DataFrame([{'k': k, 'plateau Var_c(pi)': pi.var(), 'A(0)': (pi*(1-pi)).mean()}
              for k in (0.01, 1.0, 10.0)])

,k,plateau Var_c(pi),A(0)
0,0.01,0.09,0.16
1,1.00,0.09,0.16
2,10.00,0.09,0.16


### U3: a common induction shift rescales the plateau but cannot create one

In [3]:
pd.DataFrame([{'logit shift': d, 'pi_c shifted': np.round(expit(logit(pi)+d), 3),
               'Var (clones differ)': expit(logit(pi)+d).var(),
               'Var (clones identical)': expit(logit(np.array([.5,.5]))+d).var()}
              for d in (0, .5, 1, 2)])

,logit shift,pi_c shifted,Var (clones differ),Var (clones identical)
0,0.0,"[0.2, 0.8]",0.090000,0.0
1,0.5,"[0.292, 0.868]",0.083076,0.0
2,1.0,"[0.405, 0.916]",0.065323,0.0
3,2.0,"[0.649, 0.967]",0.025359,0.0


## 2. Separating occupancy from switching rate (E10-focus, 20 clones, 10 replicates)
OCC and FROZE share an occupancy distribution and differ only in timescale. FROZE sits below the
identifiability window, where the point estimates break but the likelihood-ratio tests stay calibrated.

In [4]:
f = 'e10_focus.csv' if have('e10_focus.csv') else 'e10_occupancy_vs_rate.csv'
d = pd.read_csv(T+f)
cols = [c for c in ['true_V_pi','V_pi','true_V_logk','V_logk','n_clones_k_identified',
                    'lrt_pi_p','lrt_k_p'] if c in d.columns]
display(d.groupby('mech')[cols].median().round(4))
display(d.assign(rejects_shared_pi=d.lrt_pi_p<.05, rejects_shared_k=d.lrt_k_p<.05)
         .groupby('mech')[['rejects_shared_pi','rejects_shared_k']].mean().round(3))

,true_V_pi,V_pi,true_V_logk,V_logk,n_clones_k_identified,lrt_pi_p,lrt_k_p
mech,,,,,,,
BOTH,0.0272,0.0286,0.9472,0.1719,17.0,0.0000,0.0001
FROZE,0.0325,0.1782,0.0000,3.1436,12.0,0.0000,0.9576
OCC,0.0329,0.0304,0.0000,0.0000,19.0,0.0000,0.6554
RATE,0.0000,0.0048,1.0135,0.1087,16.5,0.2507,0.0000


,rejects_shared_pi,rejects_shared_k
mech,,
BOTH,1.0,0.9
FROZE,1.0,0.0
OCC,1.0,0.0
RATE,0.3,1.0


In [5]:
if have('e10_nonstationary_induction.csv'):
    display(pd.read_csv(T+'e10_nonstationary_induction.csv').groupby('shift')[
        ['true_V_pi_base','true_V_pi_shifted','V_pi']].median().round(4))

## 3. The sampling-design result (U5)

In [6]:
pd.read_csv(T+'e11_optimal_depth.csv').round(4)

,k,pi,L,kL,I_logk,L_star_theory
0,0.1,0.3,4.0679,0.4068,0.1393,5.00
1,0.1,0.5,4.0679,0.4068,0.1619,5.00
2,0.5,0.3,0.8093,0.4046,0.1393,1.00
3,0.5,0.5,0.8093,0.4046,0.1619,1.00
4,2.0,0.3,0.2107,0.4214,0.1393,0.25
5,2.0,0.5,0.1970,0.3940,0.1619,0.25


In [7]:
if have('e11_equal_budget.csv'):
    e = pd.read_csv(T+'e11_equal_budget.csv')
    cols = [c for c in ['true_V_pi','V_pi','true_V_logk','V_logk','n_clones_k_identified'] if c in e.columns]
    display(e.groupby(['design','C','cells_per_clone','depth'])[cols].median().round(4))

,,,,true_V_pi,V_pi,true_V_logk,V_logk,n_clones_k_identified
design,C,cells_per_clone,depth,,,,,
balanced,20,64,6,0.0318,0.0365,0.6332,0.0697,18.0
few_clones_deep,10,128,7,0.0335,0.0340,0.9046,0.7468,9.0
many_clones_shallow,40,32,5,0.0298,0.0406,0.8585,0.3774,30.5
very_few_very_deep,5,256,8,0.0280,0.0213,0.5579,0.2466,5.0


### Identifiability windows of two real recording technologies

In [8]:
s = pd.read_csv(T+'m3_depth_spectra_summary.csv')
s.groupby(['system','dataset'])[['n_leaves','depth_max','k_lo_identifiable',
                                 'k_hi_identifiable','window_decades']].median().round(3)

n_leaves  depth_max  k_lo_identifiable  k_hi_identifiable  window_decades
system      dataset                                                                           
Cassiopeia  m100k        35.0        9.0              0.062              0.125           0.204
            m10k         93.0        8.0              0.071              0.100           0.176
            m30k        400.0       13.0              0.042              0.051           0.085
            m5k          76.0       14.0              0.038              0.071           0.243
macsGESTALT mouse1       73.5        4.0              0.167              0.250           0.176
            mouse2       63.5        3.5              0.250              0.500           0.176

## 4. Simeonov re-analysis with the per-clone estimator

In [9]:
if have('m3_real_variance_components.csv'):
    display(pd.read_csv(T+'m3_real_variance_components.csv').round(4))

,mouse,scope,lrt_pi_stat,lrt_pi_p,lrt_k_stat,lrt_k_p,n_clones,V_pi_raw,V_pi_noise,V_pi,mean_pi,n_clones_k_identified,V_logk_raw,V_logk_noise,V_logk,median_k
0,1,all_sites,106.5675,0.0,50.1075,0.0000,11,0.0448,0.0135,0.0314,0.3116,7,0.2736,0.6028,0.0000,1.0891
1,1,PT_only,74.8721,0.0,51.1113,0.0000,8,0.0657,0.0131,0.0526,0.3006,5,0.3180,0.5262,0.0000,0.8383
2,2,all_sites,92.9706,0.0,58.8920,0.0001,24,0.0546,0.0149,0.0397,0.4922,11,0.5042,1.0773,0.0000,1.4443
3,2,PT_only,145.7683,0.0,8.1720,0.9627,18,0.0495,0.0149,0.0346,0.5024,8,2.3088,1.1117,1.1971,1.7296


In [10]:
if have('m3_real_perclone.csv'):
    p = pd.read_csv(T+'m3_real_perclone.csv')
    display(p[['mouse','clone','n','pi','pi_lo','pi_hi','k','k_at_bound']].round(3).head(30))
    print('clones with an identified rate:', int((~p.k_at_bound).sum()), 'of', len(p))

,mouse,clone,n,pi,pi_lo,pi_hi,k,k_at_bound
0,1,1,7968,0.683,NaN,NaN,0.598,False
1,1,2,1025,0.439,0.42,0.46,1.089,False
2,1,3,821,0.049,0.06,0.06,1.880,False
3,1,4,105,0.120,0.06,0.38,0.766,False
4,1,5,238,0.249,0.18,0.34,1.371,False
5,1,6,38,0.414,0.26,0.78,0.677,False
6,1,7,31,0.161,0.06,0.82,27.385,True
7,1,8,27,0.000,0.02,0.82,34.025,True
8,1,10,45,0.374,0.14,0.50,2.882,False
9,1,11,59,0.452,0.30,0.66,0.854,False


clones with an identified rate: 32 of 61


### Held-out metastatic site: prediction ladder

In [11]:
if have('m3_real_heldout_ladder.csv'):
    h = pd.read_csv(T+'m3_real_heldout_ladder.csv')
    ec = [c for c in h.columns if c.startswith('err_')]
    display(h.groupby('mouse')[ec].mean().round(4))
    for m in sorted(h.mouse.unique()):
        g = h[(h.mouse==m) & (h.n < h[h.mouse==m].n.max())]
        if len(g): print(f'mouse {m} excluding dominant clone:', g[ec].mean().round(4).to_dict())

,err_p_grand,err_p_pt_mean,err_p_clone_mean,err_p_tree_pi
mouse,,,,
1,0.2956,0.1679,0.0770,0.1048
2,0.1768,0.2176,0.1845,0.1012


mouse 1 excluding dominant clone: {'err_p_grand': 0.316, 'err_p_pt_mean': 0.195, 'err_p_clone_mean': 0.0931, 'err_p_tree_pi': 0.0993}
mouse 2 excluding dominant clone: {'err_p_grand': 0.2138, 'err_p_pt_mean': 0.242, 'err_p_clone_mean': 0.2291, 'err_p_tree_pi': 0.1251}


### Leave-one-clone-out stability of the occupancy component

In [12]:
if have('m3_real_leave_one_clone_out.csv'):
    l = pd.read_csv(T+'m3_real_leave_one_clone_out.csv')
    display(l.groupby('mouse').V_pi.describe()[['count','min','50%','max']].round(4))

,count,min,50%,max
mouse,,,,
1,11.0,0.0194,0.0329,0.0363
2,24.0,0.0320,0.0410,0.0430


### PATH-style baseline: one autocorrelation number per site

In [13]:
if have('m3_real_path_baseline.csv'):
    display(pd.read_csv(T+'m3_real_path_baseline.csv').round(4))

,mouse,site,n_cells,path_autocorrelation,n_clones
0,1,PT,900,0.5244,28
1,1,Met,900,0.0899,6
2,1,Liver,900,0.0980,9
3,2,PT,900,0.1090,66
4,2,Met,900,0.0038,6
5,2,Liver,900,0.0072,11


### Independent-gene validation (held-out programmes, adjusted for clone size, depth, mito, compartment)

In [14]:
if have('m3_real_gene_validation.csv'):
    g = pd.read_csv(T+'m3_real_gene_validation.csv')
    display(g[['mouse','prog','n_clones','rho_pi','p_pi','rho_pi_adj','p_pi_adj']].round(3))

,mouse,prog,n_clones,rho_pi,p_pi,rho_pi_adj,p_pi_adj
0,1,prolif,11,0.109,0.750,-0.073,0.832
1,1,glyco,11,0.145,0.670,-0.255,0.450
2,1,hypoxia,11,-0.082,0.811,-0.045,0.894
3,1,s100,11,-0.255,0.450,-0.827,0.002
4,1,stress,11,-0.927,0.000,-0.673,0.023
5,1,stem,11,-0.500,0.117,-0.645,0.032
6,1,ifn,11,0.300,0.370,-0.018,0.958
7,1,adhesion,11,-0.191,0.574,-0.464,0.151
8,1,chromatin,11,0.618,0.043,0.600,0.051
9,2,prolif,24,-0.392,0.058,-0.149,0.488
